# How Many Segments Can Marketing Actually Act On? — Assignment

**Week 4 · Assignment 4** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 4*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week4/assignments/week04_segment_count.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lesson 12 ("Segmenting Real Customers with RFM") —
same dataset, same RFM features, a different constraint.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_online_retail
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range, check_shape, check_type

## Before You Start

- **Format — same shape every week:** **A. Extend** (40 pts, scaffolded
  fill-ins on the week's mechanic) → **B. Decide** (30 pts, one modeling
  decision with a number and a justification) → **C. Recommend** (30 pts, a
  stakeholder-facing writeup). Edit only the `# YOUR CODE HERE` blocks and
  the *\<placeholder\>* prose — everything else is given.
- **Self-checks are feedback, not the grade.** The `assert` cells after each
  Part A fill-in are checkpoints: green output means you're on track. They
  don't award points on their own.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus). Parts B and C are graded on
  judgment against *your own* results, so ground your answer in the numbers
  your own run produced.

## Summary of Answers

*Fill this in last, after finishing Parts B and C below.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** | _Your answer here._ |
| **C — Recommend** | _Your answer here._ |

## 1. Business Problem

Lesson 12 built four RFM segments and recommended four different
campaigns — one per segment. But designing and running a campaign has a
real fixed cost, and marketing can only staff **3 campaigns this quarter**,
not 4. That cap wasn't part of Lesson 12's decision. This assignment adds
it back in: how many segments should the business actually build, and
which one needs the win-back budget?

**Business question:** under a hard cap of 3 campaigns this quarter, how
many customer segments (K) should the business build, and which segment
deserves the win-back offer?

**Why it matters:** four statistically distinct segments are worthless if
marketing can only execute two or three campaigns — the K that wins isn't
the one with the best fit, it's the best fit that's actually affordable to
act on.

## 2. Data — Part A: Extend

*Same dataset as Lesson 8/12. Fill in the RFM aggregation — same recipe
Lesson 12 used.*

In [ ]:
df = load_online_retail()
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# Recency is measured relative to the day after the dataset's last transaction.
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
df.head()

In [ ]:
# TODO: one row per CustomerID — Recency (days since that customer's most
# recent InvoiceDate), Frequency (number of unique InvoiceNo), Monetary
# (sum of TotalPrice). Same recipe as Lesson 12.
# YOUR CODE HERE

print(f"{len(rfm):,} customers")
rfm.describe()

In [ ]:
check_equal(len(rfm), df["CustomerID"].nunique(), "one row per customer")
check_equal(list(rfm.columns), ["Recency", "Frequency", "Monetary"], "rfm has exactly the three RFM columns, in order")
check_equal((rfm["Recency"] >= 0).all(), True, "no negative Recency values")

### Data Prep

*Same as Lesson 12 — log-transform the skewed Frequency/Monetary columns,
then standardize. Given below, nothing to edit.*

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)
X = StandardScaler().fit_transform(rfm_log)

## 3. Model — Part A: Extend

*No supervised baseline here — no label to beat. Sweep k, same as Lesson
10/12, and record inertia + silhouette at each candidate.*

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 9)

# TODO: for each k in ks, fit KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED)
# on X, and append its .inertia_ to inertias and its silhouette_score to sil_scores.
# YOUR CODE HERE

print(dict(zip(ks, [round(s, 3) for s in sil_scores])))

In [ ]:
check_equal(len(sil_scores), len(list(ks)), "one silhouette score per candidate k")
check_in_range(min(sil_scores), -1.0, 1.0, "silhouette scores are valid")

## 4. Evaluate — Part A: Extend

*Which k does silhouette alone prefer? Read it off the sweep above.*

In [ ]:
# TODO: the k (from `ks`) with the highest silhouette score.
# YOUR CODE HERE

print(f"Silhouette's numeric peak: k={best_silhouette_k}")

In [ ]:
check_equal(best_silhouette_k, 2, "matches Lesson 12's own observation that silhouette peaks at k=2")

Silhouette alone says k=2. The campaign cap allows k=2 or k=3, so before
deciding, check what each candidate's segments actually *are*. Three checks:
how they line up with Lesson 12's four named segments, whether anyone who
ordered last month would get the win-back offer, and where the revenue sits.

In [ ]:
# Lesson 12's four segments, refit on the same X with the same seed, with its names
lesson12_names = {1: "Champions", 0: "Core Regulars", 3: "Promising New", 2: "Lapsed / At-Risk"}
lesson12 = pd.Series(
    KMeans(n_clusters=4, n_init=10, random_state=RANDOM_SEED).fit_predict(X), index=rfm.index
).map(lesson12_names).rename("Lesson 12")

# The two candidates the campaign cap allows, one Series of segment labels each
candidates = {
    k: pd.Series(KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit_predict(X), index=rfm.index, name=f"k={k}")
    for k in (2, 3)
}

In [ ]:
# TODO: for each candidate k, build a cross-tab of its segments against Lesson 12's:
# a table with one row per candidate segment and one column per Lesson 12 segment,
# where each cell counts the customers who fall in both. pd.crosstab(rows, columns)
# builds it from two label Series. Store each table in crosstabs[k].
crosstabs = {}
# YOUR CODE HERE

for k, table in crosstabs.items():
    print(f"k={k}")
    display(table)

In [ ]:
check_shape(crosstabs[3], (3, 4), "k=3 table: 3 segments x Lesson 12's 4")
check_equal(int(crosstabs[2].to_numpy().sum()), len(rfm), "every customer counted exactly once")

In [ ]:
# TODO: for each candidate k, find its win-back segment (the one with the highest
# mean Recency), then count the customers in it who ordered within the last 30
# days (Recency <= 30). Store each count in recent_in_winback[k].
recent_in_winback = {}
# YOUR CODE HERE

print("Customers who ordered in the last 30 days, in each win-back segment:", recent_in_winback)

In [ ]:
check_equal(recent_in_winback, {2: 433, 3: 134}, "recent buyers in each win-back segment")

In [ ]:
# TODO: for each candidate k, each segment's share of total revenue: the sum of
# Monetary within the segment divided by the sum of Monetary overall. Store a
# Series (one value per segment) in revenue_share[k].
revenue_share = {}
# YOUR CODE HERE

for k, share in revenue_share.items():
    print(f"k={k}:", share.round(3).to_dict())

In [ ]:
check_close(revenue_share[3].sum(), 1.0, "k=3 revenue shares add up to 100%")
check_close(revenue_share[3].max(), 0.680, "k=3's top segment's share of revenue", rtol=1e-2)

## 5. Interpret — Part B: Decide

Marketing can staff **at most 3 campaigns** this quarter — so K must be 2
or 3, not more. Both satisfy the constraint; the decision is which one. Silhouette has a view,
but so does the Evaluate evidence above — weigh both.

*Fit k-means at your chosen K (2 or 3), assign `rfm["Segment"]`, and report
its silhouette score.*

In [ ]:
# TODO: pick chosen_k (2 or 3), fit KMeans at that k, and assign the labels
# to rfm["Segment"].
# YOUR CODE HERE

chosen_silhouette = silhouette_score(X, rfm["Segment"])
print(f"Chosen K={chosen_k}  silhouette={chosen_silhouette:.3f}")

In [ ]:
check_equal(chosen_k in (2, 3), True, "chosen_k satisfies the <=3-campaign constraint")
check_equal(rfm["Segment"].nunique(), chosen_k, "the fitted model actually produced chosen_k segments")

**Your answer:** *\<the K you chose\>*

**Justification (3–5 sentences):** _Your answer here._

## 6. Recommend — Part C: Recommend

*Stakeholder-facing. Write for the marketing lead who has to staff these
campaigns, not for a grader — reference your own Part B segments.*

In [ ]:
segment_sizes = rfm["Segment"].value_counts().sort_index()
profile = rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].mean().round(1)
profile["n_customers"] = segment_sizes
profile["revenue_share"] = revenue_share[chosen_k].round(3)
profile

**Recommendation:** _Your answer here._

**What would change this recommendation:** _Your answer here._

**Limits:** _Your answer here._

## Rubric

*Same four criteria, same point split, every week — 100 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's chosen K is defensible given the campaign cap and *your own* Evaluate evidence — silhouette, the cross-tab against Lesson 12, recent buyers in the win-back segment, and revenue share. |
| **Justification** | 20 | Part B's reasoning (10 pts) engages with your own numbers; Part C's recommendation (10 pts) follows logically from your own Part B answer. |
| **Communication** | 20 | Part C reads as a stakeholder memo — clear, actionable, and honest about limits. |